# Conv1D Autoencoder with SWaT 2020

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from pathlib import Path

In [ ]:
clean_folder = Path("SWaT.A7_June_2020_clean")
file_paths = sorted(clean_folder.glob("*.csv"))

print("Archivos encontrados:", len(file_paths))
for fp in file_paths:
    print(fp)

In [ ]:
def load_clean_csv(path):
    df = pd.read_csv(path)
    
    timestamps = pd.to_datetime(df["t_stamp"], errors="coerce")
    df = df.drop(columns=["t_stamp"])
    
    df = df.ffill().bfill()
    
    return df, timestamps

In [ ]:
dfs = []
timestamps_list = []

for fp in file_paths:
    df_i, ts_i = load_clean_csv(fp)
    dfs.append(df_i)
    timestamps_list.append(ts_i)

print("Número de archivos cargados:", len(dfs))
print("Shape archivo 1:", dfs[0].shape)

In [ ]:
base_columns = dfs[0].columns.tolist()

for i, df_i in enumerate(dfs):
    assert df_i.columns.tolist() == base_columns, f"Columnas distintas en archivo {i}"

print("Todos los archivos tienen las mismas columnas.")
print("Número de variables:", len(base_columns))

In [ ]:
def temporal_split(df, train_ratio=0.6, val_ratio=0.2):
    n = len(df)
    
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    
    train_df = df.iloc[:train_end].copy()
    val_df   = df.iloc[train_end:val_end].copy()
    test_df  = df.iloc[val_end:].copy()
    
    return train_df, val_df, test_df

In [ ]:
train_parts = []
val_parts = []
test_parts = []

for i, df_i in enumerate(dfs):
    train_i, val_i, test_i = temporal_split(df_i, train_ratio=0.6, val_ratio=0.2)
    
    train_parts.append(train_i)
    val_parts.append(val_i)
    test_parts.append(test_i)
    
    print(f"Archivo {i+1}:")
    print("  train:", train_i.shape)
    print("  val:  ", val_i.shape)
    print("  test: ", test_i.shape)

train_df = pd.concat(train_parts, axis=0, ignore_index=True)
val_df   = pd.concat(val_parts, axis=0, ignore_index=True)
test_df  = pd.concat(test_parts, axis=0, ignore_index=True)

print("\nShapes finales:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
scaler = StandardScaler()

train_scaled = scaler.fit_transform(train_df)
val_scaled   = scaler.transform(val_df)
test_scaled  = scaler.transform(test_df)

train_scaled = pd.DataFrame(train_scaled, columns=train_df.columns)
val_scaled   = pd.DataFrame(val_scaled, columns=val_df.columns)
test_scaled  = pd.DataFrame(test_scaled, columns=test_df.columns)

print(train_scaled.shape, val_scaled.shape, test_scaled.shape)

In [ ]:
def create_windows(data, window_size=60, stride=5):
    windows = []
    
    for i in range(0, len(data) - window_size, stride):
        window = data[i:i+window_size]
        windows.append(window)
    
    return np.array(windows)

In [ ]:
L = 60
stride = 5

train_windows = create_windows(train_scaled.values, window_size=L, stride=stride)
val_windows   = create_windows(val_scaled.values, window_size=L, stride=stride)
test_windows  = create_windows(test_scaled.values, window_size=L, stride=stride)

print("Train windows:", train_windows.shape)
print("Validation windows:", val_windows.shape)
print("Test windows:", test_windows.shape)

In [ ]:
X_train = np.transpose(train_windows, (0, 2, 1))
X_val   = np.transpose(val_windows, (0, 2, 1))
X_test  = np.transpose(test_windows, (0, 2, 1))

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
class Conv1dAutoencoder(nn.Module):
    def __init__(self, input_channels=60):
        super(Conv1dAutoencoder, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=input_channels, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),   # seq_len: 60 -> 30
            
            nn.Conv1d(in_channels=32, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2)    # seq_len: 30 -> 15
        )
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(in_channels=16, out_channels=32, kernel_size=2, stride=2),  # 15 -> 30
            nn.ReLU(),
            
            nn.ConvTranspose1d(in_channels=32, out_channels=input_channels, kernel_size=2, stride=2),  # 30 -> 60
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

In [ ]:
input_channels = X_train.shape[1]   # debería ser 60

model = Conv1dAutoencoder(input_channels=input_channels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Device:", device)
print("Input channels:", input_channels)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30
loss_history = []

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        x = batch[0].to(device)
        
        optimizer.zero_grad()
        
        x_hat = model(x)
        loss = criterion(x_hat, x)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    epoch_loss = total_loss / len(train_loader)
    loss_history.append(epoch_loss)
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss}")

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Conv1D Autoencoder Training Loss Curve")
plt.grid(True)
plt.show()

In [ ]:
model.eval()

with torch.no_grad():
    x_val = X_val_tensor.to(device)
    x_val_hat = model(x_val)
    
    val_error = torch.mean((x_val - x_val_hat)**2, dim=(1, 2))
    val_reconstruction_errors = val_error.cpu().numpy()

In [ ]:
with torch.no_grad():
    x_test = X_test_tensor.to(device)
    x_test_hat = model(x_test)
    
    test_error = torch.mean((x_test - x_test_hat)**2, dim=(1, 2))
    test_reconstruction_errors = test_error.cpu().numpy()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(val_reconstruction_errors, bins=100)
plt.title("Distribución error de reconstrucción (validation) - Conv1D AE")
plt.xlabel("Error")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
percentiles = np.percentile(val_reconstruction_errors, [90, 95, 97.5, 99])
print("P90, P95, P97.5, P99 =", percentiles)

In [ ]:
tau_95  = np.percentile(val_reconstruction_errors, 95)
tau_975 = np.percentile(val_reconstruction_errors, 97.5)
tau_99  = np.percentile(val_reconstruction_errors, 99)

print("tau_95 =", tau_95)
print("tau_97.5 =", tau_975)
print("tau_99 =", tau_99)

In [ ]:
far_95  = np.mean(test_reconstruction_errors > tau_95)
far_975 = np.mean(test_reconstruction_errors > tau_975)
far_99  = np.mean(test_reconstruction_errors > tau_99)

print("FAR @ P95:", far_95, "->", far_95 * 100, "%")
print("FAR @ P97.5:", far_975, "->", far_975 * 100, "%")
print("FAR @ P99:", far_99, "->", far_99 * 100, "%")

In [ ]:
print("Val error mean:", np.mean(val_reconstruction_errors))
print("Test error mean:", np.mean(test_reconstruction_errors))

print("Val error median:", np.median(val_reconstruction_errors))
print("Test error median:", np.median(test_reconstruction_errors))

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(val_reconstruction_errors, bins=100, alpha=0.6, label="Validation")
plt.hist(test_reconstruction_errors, bins=100, alpha=0.6, label="Test")
plt.title("Validation vs Test Reconstruction Error - Conv1D AE")
plt.xlabel("Error")
plt.ylabel("Frecuencia")
plt.legend()
plt.show()